# Experimental Design & Statistical Quality Practice Notebook

This notebook is a hands-on companion to the module on **Experimental Design & Statistical Quality**.  
It focuses on controlled experiments and practical statistical design.

Topics covered:

1. Design of Experiments (DOE) basics  
2. Full factorial analysis  
3. Fractional factorial intuition  
4. Response Surface Methodology (RSM)  
5. Sensitivity analysis  
6. ANOVA-based factor significance  

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error
from scipy import stats

np.random.seed(42)

## 1. DOE Basics

A controlled experiment typically includes:

- **factors**: input variables we control
- **levels**: values of each factor
- **response**: output we measure
- **treatments**: combinations of factor levels
- **replication**: repeating the same treatment
- **randomization**: random run order

We first define a simple two-factor experiment.

In [ ]:
factors = {
    'Temperature': [50, 70],
    'Pressure': [10, 20],
}

treatments = pd.DataFrame(list(itertools.product(*factors.values())), columns=factors.keys())
treatments

### Randomized Run Order

Randomization helps reduce ordering bias.

In [ ]:
randomized_runs = treatments.sample(frac=1, random_state=42).reset_index(drop=True)
randomized_runs

## 2. Full Factorial Analysis

For \(k\) factors each with \(L\) levels, the total number of runs is:

$$
N = L^k
$$

We now build a full factorial design with three factors at two levels each.

In [ ]:
full_factors = {
    'A': [-1, 1],
    'B': [-1, 1],
    'C': [-1, 1],
}

full_design = pd.DataFrame(list(itertools.product(*full_factors.values())), columns=full_factors.keys())
full_design

### Simulate a Response with Main Effects and Interactions

We create a synthetic response:

$$
Y = \beta_0 + \beta_A A + \beta_B B + \beta_C C + \beta_{AB} AB + \epsilon
$$

In [ ]:
full_design['AB'] = full_design['A'] * full_design['B']
full_design['AC'] = full_design['A'] * full_design['C']
full_design['BC'] = full_design['B'] * full_design['C']
full_design['ABC'] = full_design['A'] * full_design['B'] * full_design['C']

full_design['Y'] = (
    20
    + 4.0 * full_design['A']
    + 2.5 * full_design['B']
    + 1.0 * full_design['C']
    + 3.5 * full_design['AB']
    + np.random.normal(0, 0.8, len(full_design))
)
full_design

### Estimate Main Effects

For a two-level design, a simple effect estimate for factor A is the difference between the average response at high level and low level.

In [ ]:
main_effects = {}
for col in ['A', 'B', 'C']:
    high_mean = full_design.loc[full_design[col] == 1, 'Y'].mean()
    low_mean = full_design.loc[full_design[col] == -1, 'Y'].mean()
    main_effects[col] = high_mean - low_mean

pd.DataFrame({'Factor': list(main_effects.keys()), 'Estimated Main Effect': list(main_effects.values())})

## 3. Fractional Factorial Intuition

A fractional factorial design uses a subset of the full factorial runs.

For a full \(2^3 = 8\) design, we can take a half fraction with 4 runs to reduce cost.

In [ ]:
fractional_design = full_design.loc[full_design['ABC'] == 1, ['A', 'B', 'C', 'Y']].reset_index(drop=True)
fractional_design

### Compare Number of Runs

Fractional designs are cheaper, but they may alias some effects.

In [ ]:
pd.DataFrame({
    'Design Type': ['Full Factorial', 'Half Fraction'],
    'Runs': [len(full_design), len(fractional_design)]
})

## 4. Response Surface Methodology (RSM)

A common quadratic response surface model is:

$$
Y = \beta_0 + \sum \beta_i x_i + \sum \beta_{ii} x_i^2 + \sum \beta_{ij} x_i x_j + \epsilon
$$

We create a smooth 2D response surface using two factors.

In [ ]:
x1 = np.linspace(-2, 2, 12)
x2 = np.linspace(-2, 2, 12)
grid = pd.DataFrame(list(itertools.product(x1, x2)), columns=['x1', 'x2'])

grid['y'] = (
    10
    + 2.0 * grid['x1']
    - 1.5 * grid['x2']
    - 2.5 * grid['x1']**2
    - 1.0 * grid['x2']**2
    + 1.2 * grid['x1'] * grid['x2']
    + np.random.normal(0, 0.6, len(grid))
)
grid.head()

In [ ]:
X_rsm = grid[['x1', 'x2']]
y_rsm = grid['y']

rsm_model = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
rsm_model.fit(X_rsm, y_rsm)
y_pred_rsm = rsm_model.predict(X_rsm)

pd.DataFrame({
    'Metric': ['R2', 'RMSE'],
    'Value': [r2_score(y_rsm, y_pred_rsm), mean_squared_error(y_rsm, y_pred_rsm) ** 0.5]
})

In [ ]:
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(grid['x1'], grid['x2'], grid['y'])
ax.set_title('Response Surface Data')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('y')
plt.show()

## 5. Sensitivity Analysis

Sensitivity analysis studies how strongly the response changes when a factor changes.

A simple local sensitivity view can be obtained by varying one factor while holding the others fixed.

In [ ]:
x1_line = np.linspace(-2, 2, 100)
x2_fixed = 0.0
sens_df = pd.DataFrame({'x1': x1_line, 'x2': x2_fixed})
sens_pred = rsm_model.predict(sens_df)

plt.figure(figsize=(7, 4))
plt.plot(x1_line, sens_pred)
plt.title('Sensitivity of Response to x1 (x2 fixed at 0)')
plt.xlabel('x1')
plt.ylabel('Predicted Response')
plt.show()

### Approximate Sensitivity Ranking from Full Factorial Effects

In [ ]:
effect_rank = pd.DataFrame({
    'Factor': list(main_effects.keys()),
    'Absolute Effect': [abs(v) for v in main_effects.values()]
}).sort_values('Absolute Effect', ascending=False)
effect_rank

## 6. ANOVA-Based Factor Significance

ANOVA is commonly used to test whether factor effects are statistically significant.

The basic F ratio is:

$$
F = \frac{MS_{factor}}{MS_{error}}
$$

We simulate replications so that ANOVA can estimate experimental error.

In [ ]:
replicated_rows = []
for _, row in treatments.iterrows():
    temp = row['Temperature']
    press = row['Pressure']
    for rep in range(5):
        y_val = (
            40
            + 0.45 * temp
            + 0.80 * press
            + 0.03 * temp * press
            + np.random.normal(0, 2.0)
        )
        replicated_rows.append((temp, press, rep + 1, y_val))

anova_df = pd.DataFrame(replicated_rows, columns=['Temperature', 'Pressure', 'Replication', 'Response'])
anova_df.head()

### One-Way ANOVA for Temperature Effect

Here we test whether mean response differs between temperature levels.

In [ ]:
temp_groups = [group['Response'].values for _, group in anova_df.groupby('Temperature')]
f_temp, p_temp = stats.f_oneway(*temp_groups)
pd.DataFrame({'Statistic': ['F (Temperature)', 'p-value'], 'Value': [f_temp, p_temp]})

### One-Way ANOVA for Pressure Effect

In [ ]:
press_groups = [group['Response'].values for _, group in anova_df.groupby('Pressure')]
f_press, p_press = stats.f_oneway(*press_groups)
pd.DataFrame({'Statistic': ['F (Pressure)', 'p-value'], 'Value': [f_press, p_press]})

### Visual Comparison of Factor Levels

In [ ]:
temp_means = anova_df.groupby('Temperature')['Response'].mean()
press_means = anova_df.groupby('Pressure')['Response'].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(temp_means.index.astype(str), temp_means.values)
axes[0].set_title('Mean Response by Temperature')
axes[0].set_xlabel('Temperature')
axes[0].set_ylabel('Mean Response')

axes[1].bar(press_means.index.astype(str), press_means.values)
axes[1].set_title('Mean Response by Pressure')
axes[1].set_xlabel('Pressure')
axes[1].set_ylabel('Mean Response')

plt.tight_layout()
plt.show()

## 7. Small Summary Table

This table gathers the main outputs from the notebook.

In [ ]:
summary = pd.DataFrame({
    'Measure': [
        'Full factorial runs',
        'Fractional factorial runs',
        'RSM R2',
        'RSM RMSE',
        'Largest estimated main effect',
        'Temperature ANOVA p-value',
        'Pressure ANOVA p-value'
    ],
    'Value': [
        len(full_design),
        len(fractional_design),
        r2_score(y_rsm, y_pred_rsm),
        mean_squared_error(y_rsm, y_pred_rsm) ** 0.5,
        effect_rank.iloc[0]['Factor'],
        p_temp,
        p_press
    ]
})
summary

## 8. Mini Exercises

Try these on your own:

1. Add a fourth factor to the full factorial design and calculate the new number of runs.  
2. Change the random seed and compare the randomized run order.  
3. Create a quarter-fraction design and compare cost vs information.  
4. Fit a higher-order response surface and compare the error metrics.  
5. Add stronger interaction effects and examine how the estimated effects change.  
6. Increase the number of replications and inspect how the ANOVA p-values behave.

These exercises are especially useful for students learning controlled experiments, fair comparisons, and statistical quality in scientific work.